# AICE Professional 주피터라이트 실전 모의고사 10회 — 공개데이터(중력 이상 암석 종류 · 광상 기록 주 광종 · 중력지도 지형구)

## [ 필독 ] AICE Professional 모의고사 유의사항

- 공식 샘플문항(가공식품 카테고리 분류)과 **같은 시험지 형식**으로 만든 모의고사입니다. 3문항, 180분 기준입니다.
- 공식 시험과 같은 **Tabular / Text / Image 3유형**입니다. 2번 Text 문항은 공식 샘플(상품명 → 카테고리)과 같은 텍스트 분류입니다.
- 본인 스스로 모델을 만들고 **달성 목표 이상의 성능**이 나오도록 개발해야 합니다. 정답(테스트 라벨)은 제공되지 않습니다.
  대신 이 노트북 **맨 마지막 셀**(전체 자동 채점, 정답 키 인코딩)로 제출 파일을 스스로 채점할 수 있습니다(연습용).

## 사용할 수 있는 라이브러리 (주피터라이트 판)

이 모의고사는 **브라우저 주피터라이트(Pyodide · Python 3.14)** 에서 설치 없이 끝까지 풀 수 있도록 라이브러리를 제한했습니다.
필요한 라이브러리는 주피터라이트에 **미리 설치**되어 있습니다. 맨 위 '브라우저 실습' 셀은 확인만 하고 바로 끝납니다(Colab·로컬 Jupyter 에서는 아무 일도 하지 않습니다).

| 구분 | 쓸 수 있음 | 쓰지 않음(브라우저에서 안 됨) |
|---|---|---|
| 데이터·그림 | numpy · pandas · matplotlib · seaborn · Pillow | — |
| 머신러닝 | scikit-learn · LightGBM · XGBoost · imbalanced-learn · joblib | CatBoost |
| 텍스트 | scikit-learn 의 CountVectorizer · TfidfVectorizer · 내장 영어 불용어, **NLTK**(불용어·토큰화·어간 추출, 데이터는 `/drive/nltk_data` 에 미리 들어 있음) | 형태소 분석기 · BERT 등 사전학습 언어 모델 |
| 딥러닝 | `tensorflow.keras` 문법(TensorFlow.js 로 실행되는 대체 구현체, 미리 설치): `Sequential`·함수형(`Input` → 층 → `Model`) · Dense · Dropout · BatchNormalization · Conv2D · MaxPooling2D · GlobalAveragePooling2D · Embedding · LSTM · GRU · 콜백 3종(EarlyStopping · ModelCheckpoint · ReduceLROnPlateau) · `load_img` · `img_to_array` · `image_dataset_from_directory` · `ImageDataGenerator` · `model.save` / `load_model` · **MobileNet v1 전이학습**(`alpha=0.25`, 224×224, `weights='imagenet'` 만) | PyTorch · 다른 사전학습 모델(MobileNetV2/V3 · ResNet · EfficientNet 등) · 사용자 정의 층·손실 · GPU |

- 이미지는 64×64 정도로 줄여 작은 CNN 을 직접 학습하거나, MobileNet v1(alpha 0.25)로 특징을 뽑아 분류기를 학습하세요(전이학습). 브라우저 TensorFlow.js 는 CPython TensorFlow 와 수치가 조금 달라 딥러닝 점수가 약간 다르게 나올 수 있습니다.
- 학습이 오래 걸리면 epoch 를 줄이거나 조기 종료(EarlyStopping)를 쓰세요. 최신 Chrome·Edge 를 권장합니다.

## 시간 배분(180분)

| 문항 | 권장 시간 | 먼저 할 일 |
|---|---|---|
| 1. Tabular (30점) | 50분 | `info()`·`describe()` 로 자료형·결측 확인 → 기본 모델로 제출 파일부터 만들기 |
| 2. Text (35점) | 60분 | 라벨 분포·문장 길이 확인 → TF-IDF + 선형 모델로 기준 점수 |
| 3. Image (35점) | 60분 | 압축 풀기·폴더 구조 확인 → 작은 CNN 으로 기준 점수 |
| 마무리 | 10분 | 제출 파일 6개 이름·열 구조 확인 → 맨 마지막 채점 셀 실행 |

## 시험 자원 가이드

| 구분 | 공식 시험 | 이 모의고사 |
|---|---|---|
| IDE | CPU 8 core · Memory 32G · GPU 1개(VRAM 16GB) | **브라우저 주피터라이트**(설치 없음, CPU) — Colab·로컬 Jupyter 에서도 그대로 실행됨 |
| SW | Python 3.10 · TensorFlow 2.13 · PyTorch 2.1 | Python 3.14(Pyodide) · scikit-learn · LightGBM · XGBoost · Keras 문법(TensorFlow.js) — 위 표의 라이브러리만 |

## 유의 사항

- 막히면 문항 아래 접혀 있는 **힌트 1·2·3단계**를 순서대로 펼쳐 보세요. 단계가 올라갈수록 정답에 가까워집니다.
- 제출 파일명은 **고정**입니다: `answer_1~3` 모델 파일과 `answer_1~3.csv`.


문제에 제시되는 가이드는 필수적으로 준수하여 주시기 바랍니다. 이를 준수하지 않을 경우 채점 제한 또는 0점 처리될 수 있으니
각별히 유의해 주시기 바랍니다.

#### **[학습 정보]**
- 학습 목표 : Tabular · Text · Image 예측 모델 3종을 만들어 달성 목표 이상의 예측 파일을 제출하고, 제출한 코드로 같은 모델이 재현되게 할 수 있다
- AICE 역량 : PT1·PT2 표 데이터 / PX1·PX2 텍스트 / PI1·PI2 이미지 / PS1 제출 파일 형식·재현성
- 제출 파일 : `answer_1~3` 모델 파일(.pkl · .h5 등)과 `answer_1~3.csv` — **파일명은 고정**입니다(이름을 바꾸면 채점되지 않음)

| 문항 | 문제 | 달성 목표 | 역량 | 기술 | 예상 시간 |
|---|---|---|---|---|---|
| 1. Tabular | 중력 이상으로 지표 암석 종류 분류(4종) | 정확도 0.53 이상 | PT1 표 데이터 전처리·피처 엔지니어링, PT2 표 데이터 모델 학습·평가 | pandas, scikit-learn(LogisticRegression, StandardScaler, RandomForestClassifier, StratifiedKFold), LightGBM, joblib | 50분 |
| 2. Text | 광상 조사 기록으로 주 광종 분류(6종) | 정확도 0.59 이상 | PX1 텍스트 정제·벡터화, PX2 텍스트 분류 모델 학습·평가 | pandas, scikit-learn(CountVectorizer, TfidfVectorizer, MultinomialNB, LinearSVC), joblib | 60분 |
| 3. Image | 중력지도 조각으로 지형구 분류(6종) | 정확도 0.58 이상 | PI1 이미지 로딩·전처리, PI2 이미지 분류 모델(CNN·전이학습) | pandas, Keras(TensorFlow.js) | 60분 |

In [ ]:
# (브라우저 실습) 주피터라이트에는 tensorflow(TensorFlow.js 대체 구현체)·scikit-learn·LightGBM 등이 미리 설치되어 있어 이 셀은 확인만 합니다.
# Colab·로컬 Jupyter 에서는 아무 일도 하지 않습니다.
import os, sys, importlib.util
if 'pyodide' in sys.modules:
    os.environ['AICE_IMG_DIR'] = '/tmp/aice_images/' + os.path.basename(os.getcwd())   # 이미지 압축은 메모리 폴더에 풂(빠름)
    if not ('tensorflow' in sys.modules or importlib.util.find_spec('tensorflow')):
        # 미리 설치되지 않은 사이트(패치 전 주피터라이트)에서만: 같은 폴더의 tfjs_keras.py 를 불러옵니다
        sys.path[:0] = [os.getcwd(), '/drive']
        import tfjs_keras
        await tfjs_keras.load()
print('준비 완료 — 다음 셀로 넘어가세요.')

준비 완료 — 다음 셀로 넘어가세요.


## 1. Tabular 문제 - 중력 이상으로 지표 암석 종류 분류(4종) (30점)

<sub>학습 목표: 중력 이상으로 지표 암석 종류 분류(4종) 모델로 테스트 정확도 0.53 이상을 달성할 수 있다 | 역량: PT1 표 데이터 전처리·피처 엔지니어링, PT2 표 데이터 모델 학습·평가 | 기술: pandas, scikit-learn(LogisticRegression, StandardScaler, RandomForestClassifier, StratifiedKFold), LightGBM, joblib | 난이도: 실전 | 예상 시간: 50분</sub>

### 문제 개요

미국 본토 격자점마다 **중력 이상(gravity anomaly) 값과 주변 창의 통계** 가 있습니다. 이 값으로 그 지점 **지표 암석 종류 4종**(퇴적암·화산암·심성암·변성암)을 분류하는 Tabular 다중 분류 문제입니다. 밀도가 높은 결정질 암석과 낮은 퇴적층은 중력 이상의 크기와 국지 변화 모양이 달라, 자원 탐사·지하 구조 판독에서 중력지도를 읽는 것과 같은 형식입니다. 학습·테스트는 **약 190km 크기의 지역 블록 단위로 나뉘어** 있어, 테스트 지점 바로 옆 지점이 학습에 들어 있지 않습니다.

제공된 훈련 데이터셋을 활용해 다중 분류(4종) 모델을 개발하고, 테스트 데이터셋의 'rocktype' 값을 예측하시기 바랍니다.

### 달성 목표

"01_test_x.csv" 파일의 테스트 데이터셋을 기준으로 **정확도(Accuracy) 53% 이상**을 달성하는 모델을 개발하시기 바랍니다.

단, 테스트 데이터의 정답(Label 값)은 별도로 제공되지 않습니다.

### 데이터 설명

**데이터 위치 및 폴더 구조** — 문제 파일과 동일한 곳에 위치

- `01_train.csv`: 훈련 데이터셋입니다. 훈련 데이터셋으로 모델을 만드세요.
- `01_test_x.csv`: 테스트 데이터셋입니다. 라벨('rocktype') 값이 제외되어 있습니다.

**Label 정보**

- 'rocktype' : Metamorphic: 변성암·결정질 기반암 / Plutonic: 심성암(화강암 등) / Sedimentary: 퇴적암 / Volcanic: 화산암

**Data 정보**

- 'id' : 행 번호
- 'region' : 지형구 대구분(Interior Plains 등 범주형)
- 'bouguer · isostatic' : 완전 부게 이상 · 등압 잔여 이상(mGal)
- 'bouguer_minus_iso' : 두 이상의 차이(지형·지각 뿌리 효과)
- '*_mean_12km / 40km / 100km' : 주변 창(반경 약 12·40·100km)의 평균
- '*_std_*' : 주변 창의 표준편차(국지 변화 크기)
- '*_resid_*' : 값 − 주변 평균(국지 잔여 이상)
- '*_gradient' : 중력 이상의 수평 경사 크기

**데이터 출처** — USGS 미국 본토 중력 격자(완전 부게 이상·등압 잔여 이상, 4km — 미국 정부 공공 저작물, 사용 제한 없음) + 북미 지질도 DS-424(HF NoraResearchLab/ds424-geologic-map-north-america · CC0)의 지표 암석 종류 — 약 12km 간격 격자점

### 데이터 예시

`01_train.csv`

| id | region | bouguer | isostatic | bouguer_minus_iso | bouguer_mean_12km | bouguer_std_12km | bouguer_resid_12km | … |
|---|---|---|---|---|---|---|---|---|
| 2 | Rocky Mountain System | -209 | 9.1 | -218.1 | -208.88 | 10.56 | -0.12 | … |
| 4 | Interior Plains | -25 | 7.5 | -32.5 | -26.06 | 4.01 | 1.06 | … |
| 5 | Intermontane Plateaus | -92 | -15.2 | -76.8 | -93.29 | 3.85 | 1.29 | … |

`01_test_x.csv`

| id | region | bouguer | isostatic | bouguer_minus_iso | bouguer_mean_12km | bouguer_std_12km | bouguer_resid_12km | … |
|---|---|---|---|---|---|---|---|---|
| 1354 | Rocky Mountain System | -205 | -17.1 | -187.9 | -208.43 | 5.73 | 3.43 | … |
| 7545 | Pacific Mountain System | -67 | -22 | -45 | -64.67 | 6.19 | -2.33 | … |
| 5896 | Interior Plains | -81 | -4.4 | -76.6 | -80.12 | 3.95 | -0.88 | … |

### 문제 해결 가이드

(1) 훈련 데이터셋을 이용하여 중력 이상으로 지표 암석 종류 분류(4종) 모델을 만드세요.
- 필요 시 추가로 함수를 만들거나 cell 을 만들어도 됩니다.
- 성능이 가장 좋은 모델을 저장하세요.
- 파일명 : **answer_1.pkl** (또는 다른 확장자의 모델 파일)

(2) 라벨('rocktype') 값이 제외된 테스트 데이터 파일을 읽어들여 예측값에 대한 컬럼을 추가하여 csv 로 저장하세요.
- 파일명 : **answer_1.csv**
- 테스트 데이터 파일을 읽고 **마지막 컬럼 뒤에 'rocktype' 컬럼 생성** 후, 모델 예측하여 예측값을 채워주세요.
- 테스트 데이터의 행 순서와 행 수를 바꾸지 마세요. 예측값은 훈련 데이터의 'rocktype' 와 같은 형식(원래 라벨 이름·단위)이어야 합니다.

(3) 본 문제지를 답안코드 작성된 상태로 저장하세요.
- 파일명 : 시스템매핑번호.ipynb (본 문제지 상단의 저장버튼을 눌러 저장)

### 참고

- `region` 은 문자열 범주이므로 원-핫 인코딩합니다. 나머지는 모두 수치(mGal)입니다.
- 학습 라벨은 퇴적암·화산암·심성암이 각 약 2,000건, 변성암이 약 800건으로 조금 불균형합니다.
- 선형 모델과 트리 기반 모델(RandomForest·LightGBM)을 **교차검증 정확도** 로 비교해 보세요. 국지 잔여 이상·표준편차 같은 특징이 중요합니다.
- 평가 지표는 **정확도(Accuracy)** 입니다. 예측값은 라벨 문자열(Sedimentary 등) 그대로 제출하세요.

### 채점 진행 방식 안내

- 1번 문항 배점 : 30점
- 시스템매핑번호.ipynb 실행 가능 여부 확인
- 제출한 코드로 실행했을 때, 제출한 모델 파일과 동일하게 재현되는지 확인
- answer_1.csv 로 제출된 예측 데이터를 정답지와 대조하여, **정확도(Accuracy) 47.7% 이상** 달성
  (공식 샘플의 '목표 70% → 채점 기준 63%' 와 같은 비율)

### 답안 작성

필요한 cell 을 추가하여 작성하세요.

In [ ]:
# 답안 작성

In [ ]:

from pathlib import Path
import shutil
import pandas as pd

# 테스트 파일명 확인
if not Path("01_test_x.csv").exists() and Path("01_test_x(1).csv").exists():
    shutil.copyfile("01_test_x(1).csv", "01_test_x.csv")

# 데이터 불러오기
train_1 = pd.read_csv("01_train.csv")
test_1 = pd.read_csv("01_test_x.csv")

print("학습 데이터:", train_1.shape)
print("테스트 데이터:", test_1.shape)

print("\n자료형 및 데이터 구조")
train_1.info()

print("\n결측치 개수")
print("학습:", train_1.isna().sum().sum())
print("테스트:", test_1.isna().sum().sum())

print("\n중복 행 개수")
print("학습:", train_1.duplicated().sum())
print("테스트:", test_1.duplicated().sum())

print("\n암석 종류별 데이터 개수")
print(train_1["rocktype"].value_counts())

print("\n지역별 데이터 개수")
print(train_1["region"].value_counts())

print("\n학습·테스트 입력 컬럼 일치 여부")
print(train_1.drop(columns=["rocktype"]).columns.equals(test_1.columns))

print("\n학습 데이터 상위 3행")
display(train_1.head(3))


학습 데이터: (6852, 26)
테스트 데이터: (1687, 25)

자료형 및 데이터 구조
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6852 entries, 0 to 6851
Data columns (total 26 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     6852 non-null   int64  
 1   region                 6852 non-null   object 
 2   bouguer                6852 non-null   float64
 3   isostatic              6852 non-null   float64
 4   bouguer_minus_iso      6852 non-null   float64
 5   bouguer_mean_12km      6852 non-null   float64
 6   bouguer_std_12km       6852 non-null   float64
 7   bouguer_resid_12km     6852 non-null   float64
 8   isostatic_mean_12km    6852 non-null   float64
 9   isostatic_std_12km     6852 non-null   float64
 10  isostatic_resid_12km   6852 non-null   float64
 11  bouguer_mean_40km      6852 non-null   float64
 12  bouguer_std_40km       6852 non-null   float64
 13  bouguer_resid_40km     6852 non-null   float64
 14  iso

,id,region,bouguer,isostatic,bouguer_minus_iso,bouguer_mean_12km,bouguer_std_12km,bouguer_resid_12km,isostatic_mean_12km,isostatic_std_12km,...,isostatic_resid_40km,bouguer_mean_100km,bouguer_std_100km,bouguer_resid_100km,isostatic_mean_100km,isostatic_std_100km,isostatic_resid_100km,isostatic_gradient,bouguer_gradient,rocktype
0,2,Rocky Mountain System,-209.0,9.1,-218.1,-208.88,10.56,-0.12,9.84,8.23,...,-11.66,-195.46,34.52,-13.54,17.69,25.32,-8.59,5.73,6.95,Sedimentary
1,4,Interior Plains,-25.0,7.5,-32.5,-26.06,4.01,1.06,6.14,3.59,...,10.94,-39.39,30.76,14.39,-6.88,29.38,14.38,1.03,1.12,Sedimentary
2,5,Intermontane Plateaus,-92.0,-15.2,-76.8,-93.29,3.85,1.29,-16.27,1.80,...,-0.03,-85.54,18.23,-6.46,-8.99,9.95,-6.21,1.00,2.55,Volcanic


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 정답과 입력 변수 구분
LABEL_1 = "rocktype"

feature_cols_1 = [
    col for col in train_1.columns
    if col not in ["id", LABEL_1]
]

X_1 = train_1[feature_cols_1].copy()
y_1 = train_1[LABEL_1].copy()
X_test_1 = test_1[feature_cols_1].copy()

# 학습 80%, 검증 20% 분리
X_train_1, X_valid_1, y_train_1, y_valid_1 = train_test_split(
    X_1,
    y_1,
    test_size=0.2,
    random_state=42,
    stratify=y_1
)

# 변수 유형 구분
cat_cols_1 = ["region"]
num_cols_1 = X_1.select_dtypes(include="number").columns.tolist()

# 선형 모델용 전처리
prep_linear_1 = ColumnTransformer([
    ("num", StandardScaler(), num_cols_1),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols_1)
])

# 트리 모델용 전처리
prep_tree_1 = ColumnTransformer([
    ("num", "passthrough", num_cols_1),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols_1)
])

print("학습 X:", X_train_1.shape)
print("검증 X:", X_valid_1.shape)
print("학습 y:", y_train_1.shape)
print("검증 y:", y_valid_1.shape)
print("테스트 X:", X_test_1.shape)

print("\n수치형 변수 개수:", len(num_cols_1))
print("범주형 변수:", cat_cols_1)

print("\n학습 데이터 클래스 비율 (%)")
print((y_train_1.value_counts(normalize=True) * 100).round(2))

print("\n검증 데이터 클래스 비율 (%)")
print((y_valid_1.value_counts(normalize=True) * 100).round(2))


학습 X: (5481, 24)
검증 X: (1371, 24)
학습 y: (5481,)
검증 y: (1371,)
테스트 X: (1687, 24)

수치형 변수 개수: 23
범주형 변수: ['region']

학습 데이터 클래스 비율 (%)
rocktype
Volcanic       29.78
Plutonic       29.57
Sedimentary    28.64
Metamorphic    12.01
Name: proportion, dtype: float64

검증 데이터 클래스 비율 (%)
rocktype
Volcanic       29.76
Plutonic       29.61
Sedimentary    28.59
Metamorphic    12.04
Name: proportion, dtype: float64


In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier
import pandas as pd

# 4-Fold 교차검증
cv_1 = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

# 비교할 모델 3개
models_1 = {
    "LogisticRegression": Pipeline([
        ("preprocess", prep_linear_1),
        ("model", LogisticRegression(max_iter=2000, random_state=42))
    ]),
    "RandomForest": Pipeline([
        ("preprocess", prep_tree_1),
        ("model", RandomForestClassifier(
            n_estimators=250,
            random_state=42,
            n_jobs=-1
        ))
    ]),
    "LightGBM": Pipeline([
        ("preprocess", prep_tree_1),
        ("model", LGBMClassifier(
            n_estimators=250,
            learning_rate=0.05,
            num_leaves=31,
            random_state=42,
            verbosity=-1,
            n_jobs=-1
        ))
    ])
}

# 교차검증 및 독립 검증
results_1 = []
fitted_models_1 = {}

for name, pipeline in models_1.items():
    cv_scores = cross_val_score(
        pipeline,
        X_train_1,
        y_train_1,
        cv=cv_1,
        scoring="accuracy",
        n_jobs=1
    )

    pipeline.fit(X_train_1, y_train_1)
    valid_pred = pipeline.predict(X_valid_1)
    valid_acc = accuracy_score(y_valid_1, valid_pred)

    fitted_models_1[name] = pipeline

    results_1.append({
        "모델": name,
        "CV Accuracy": cv_scores.mean(),
        "CV 표준편차": cv_scores.std(),
        "검증 Accuracy": valid_acc
    })

# 모델별 성능 비교표
result_df_1 = pd.DataFrame(results_1)
result_df_1 = result_df_1.sort_values(
    "CV Accuracy",
    ascending=False
).reset_index(drop=True)

display(result_df_1.round(4))


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,모델,CV Accuracy,CV 표준편차,검증 Accuracy
0,RandomForest,0.7497,0.0071,0.7914
1,LightGBM,0.7384,0.0044,0.7622
2,LogisticRegression,0.5906,0.0133,0.6193


In [ ]:

import joblib
import numpy as np
from sklearn.base import clone
from pathlib import Path

# 선정된 RandomForest 모델 복제
final_model_1 = clone(fitted_models_1["RandomForest"])

# 전체 학습 데이터로 재학습
final_model_1.fit(X_1, y_1)

# 후보 모델 저장 (정식 제출 파일은 아직 생성하지 않음)
model_path_1 = "answer_1_rf_candidate.pkl"
joblib.dump(final_model_1, model_path_1)

# 저장 모델 다시 불러오기
loaded_model_1 = joblib.load(model_path_1)

# 저장 전후 예측 비교
pred_before_1 = final_model_1.predict(X_test_1)
pred_after_1 = loaded_model_1.predict(X_test_1)

print("모델 파일:", model_path_1)
print("파일 존재:", Path(model_path_1).exists())
print("전체 학습 행 수:", len(X_1))
print("테스트 예측 행 수:", len(pred_after_1))
print("저장 전후 예측 일치:", np.array_equal(
    pred_before_1, pred_after_1
))
print("예측 클래스:", np.unique(pred_after_1))


모델 파일: answer_1_rf_candidate.pkl
파일 존재: True
전체 학습 행 수: 6852
테스트 예측 행 수: 1687
저장 전후 예측 일치: True
예측 클래스: ['Metamorphic' 'Plutonic' 'Sedimentary' 'Volcanic']


In [ ]:

import shutil
import hashlib
from pathlib import Path

# 정식 제출 파일명
final_model_path_1 = Path("answer_1.pkl")
final_csv_path_1 = Path("answer_1.csv")

# 기존 정식 제출 파일 보호
if final_model_path_1.exists() or final_csv_path_1.exists():
    raise FileExistsError("기존 정식 제출 파일이 있습니다. 덮어쓰기 전에 확인하세요.")

# 테스트 데이터에 예측 컬럼 추가
submission_1 = test_1.copy()
submission_1["rocktype"] = loaded_model_1.predict(X_test_1)

# 제출 전 검증
assert len(submission_1) == len(test_1)
assert submission_1.columns.tolist() == test_1.columns.tolist() + ["rocktype"]
assert submission_1["id"].equals(test_1["id"])
assert submission_1["rocktype"].notna().all()
assert set(submission_1["rocktype"]).issubset(set(y_1))

# 검증된 후보 모델을 정식 파일로 복사
shutil.copyfile(model_path_1, final_model_path_1)

# 예측 CSV 저장
submission_1.to_csv(final_csv_path_1, index=False)

# 저장된 파일 다시 확인
saved_csv_1 = pd.read_csv(final_csv_path_1)
saved_model_1 = joblib.load(final_model_path_1)

model_match_1 = (
    hashlib.sha256(Path(model_path_1).read_bytes()).hexdigest()
    == hashlib.sha256(final_model_path_1.read_bytes()).hexdigest()
)

csv_match_1 = saved_csv_1.equals(submission_1)
pred_match_1 = np.array_equal(
    saved_model_1.predict(X_test_1),
    saved_csv_1["rocktype"].to_numpy()
)

print("모델 파일:", final_model_path_1)
print("CSV 파일:", final_csv_path_1)
print("CSV 크기:", saved_csv_1.shape)
print("마지막 컬럼:", saved_csv_1.columns[-1])
print("모델 파일 SHA-256 일치:", model_match_1)
print("CSV 저장 전후 일치:", csv_match_1)
print("모델 예측과 CSV 일치:", pred_match_1)

print("\n예측 클래스 분포")
print(saved_csv_1["rocktype"].value_counts())

print("\n제출 CSV 상위 3행")
display(saved_csv_1.head(3))


모델 파일: answer_1.pkl
CSV 파일: answer_1.csv
CSV 크기: (1687, 26)
마지막 컬럼: rocktype
모델 파일 SHA-256 일치: True
CSV 저장 전후 일치: True
모델 예측과 CSV 일치: True

예측 클래스 분포
rocktype
Sedimentary    723
Plutonic       395
Volcanic       369
Metamorphic    200
Name: count, dtype: int64

제출 CSV 상위 3행


,id,region,bouguer,isostatic,bouguer_minus_iso,bouguer_mean_12km,bouguer_std_12km,bouguer_resid_12km,isostatic_mean_12km,isostatic_std_12km,...,isostatic_resid_40km,bouguer_mean_100km,bouguer_std_100km,bouguer_resid_100km,isostatic_mean_100km,isostatic_std_100km,isostatic_resid_100km,isostatic_gradient,bouguer_gradient,rocktype
0,1354,Rocky Mountain System,-205.0,-17.1,-187.9,-208.43,5.73,3.43,-20.56,3.04,...,0.19,-179.04,41.81,-25.96,-3.43,20.69,-13.67,1.80,2.69,Plutonic
1,7545,Pacific Mountain System,-67.0,-22.0,-45.0,-64.67,6.19,-2.33,-20.25,6.61,...,-4.99,-54.80,33.17,-12.20,-18.35,19.41,-3.65,4.31,6.95,Plutonic
2,5896,Interior Plains,-81.0,-4.4,-76.6,-80.12,3.95,-0.88,-3.70,3.28,...,-3.56,-80.77,18.89,-0.23,-5.13,14.19,0.73,1.88,2.06,Sedimentary


<details><summary>힌트 1단계 (접근 방향) - 1번 문항</summary>

클래스 분포를 확인하고, 범주형은 원-핫 인코딩합니다. 값의 크기가 다르면 선형 모델 앞에 표준화를 두고, 트리 모델과 **교차검증 정확도** 로 비교하세요.

- 이 문항의 핵심: `region` 은 문자열 범주이므로 원-핫 인코딩합니다. 나머지는 모두 수치(mGal)입니다.

</details>

<details><summary>힌트 2단계 (사용할 도구) - 1번 문항</summary>

사용할 도구: `pandas`, `joblib`, `LogisticRegression`, `StandardScaler`, `make_pipeline`, `RandomForestClassifier`, `LGBMClassifier`, `StratifiedKFold`, `cross_val_score`.

- 참고 문서: <https://joblib.readthedocs.io/en/stable/generated/joblib.dump.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html>
- 참고 문서: <https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.LGBMClassifier.html>

</details>

<details><summary>힌트 3단계 (코드 구조) - 1번 문항</summary>

코드의 **앞부분은 구조만** 보여 줍니다(`____` 자리는 가이드와 공식 문서를 보고 채우세요). **뒷부분은 주석으로 할 일만** 적었습니다 — 주석을 따라 직접 작성하세요.
```python
import ____
import ____
from sklearn.linear_model import ____
from sklearn.preprocessing import ____
from sklearn.pipeline import ____
from sklearn.ensemble import ____
from lightgbm import ____
from sklearn.model_selection import ____, ____

train = pd.read_csv('01_train.csv')
test = pd.read_csv('01_test_x.csv')
train.info(verbose=False)
LABEL = '____'

# ▷ 후보 모델을 교차검증으로 비교(점수 출력)
# ▷ 학습(검증 데이터·조기 종료)
# ▷ 모델 파일 저장 — answer_N.pkl / .h5
# ▷ 테스트 데이터 예측(행 순서 그대로)
# ▷ 예측 CSV 저장 — answer_N.csv(테스트 열 + 마지막 정답 열)
```
- 제출 파일명: `answer_1.pkl`(모델) · `answer_1.csv`(예측)

</details>

## 2. Text 문제 - 광상 조사 기록으로 주 광종 분류(6종) (35점)

<sub>학습 목표: 광상 조사 기록으로 주 광종 분류(6종) 모델로 테스트 정확도 0.59 이상을 달성할 수 있다 | 역량: PX1 텍스트 정제·벡터화, PX2 텍스트 분류 모델 학습·평가 | 기술: pandas, scikit-learn(CountVectorizer, TfidfVectorizer, MultinomialNB, LinearSVC), joblib | 난이도: 실전 | 예상 시간: 60분</sub>

### 문제 개요

광산·광상 조사 기록(영어, 평균 약 230자)으로 그 광상의 **주 광종 6종**(금·구리·납아연·우라늄·철·텅스텐)을 분류하는 자연어 처리 문제입니다. 기록에 광종 이름은 지워져(`***`) 있어, 맥석 광물(Quartz·Calcite 등)·모암(Granite·Sandstone 등)·광상 유형(Vein·Placer·Skarn 등) 같은 단서로 판단해야 합니다.

제공된 훈련 데이터셋을 활용해 텍스트 다중 분류(6종) 모델을 개발하고, 테스트 데이터셋의 'commodity' 값을 예측하시기 바랍니다.

### 달성 목표

"02_test_x.csv" 파일의 테스트 데이터셋을 기준으로 **정확도(Accuracy) 59% 이상**을 달성하는 모델을 개발하시기 바랍니다.

단, 테스트 데이터의 정답(Label 값)은 별도로 제공되지 않습니다.

### 데이터 설명

**데이터 위치 및 폴더 구조** — 문제 파일과 동일한 곳에 위치

- `02_train.csv`: 훈련 데이터셋입니다. 훈련 데이터셋으로 모델을 만드세요.
- `02_test_x.csv`: 테스트 데이터셋입니다. 라벨('commodity') 값이 제외되어 있습니다.

**Label 정보**

- 'commodity' : Copper: 구리 / Gold: 금 / Iron: 철 / Lead-Zinc: 납·아연 / Tungsten: 텅스텐 / Uranium: 우라늄

**Data 정보**

- 'record' : 광상 조사 기록 — `항목: 값 | 항목: 값` 형식(dep_type·orebody_fm·gangue·hrock_type·arock_type·structure·alteration·conc_proc·ore_ctrl·tectonic)
- 'id' : 테스트 행 번호(테스트에만)

**데이터 출처** — USGS MRDS 광물자원 데이터베이스(mrdata.usgs.gov/mrds — 미국 정부 공공 저작물, 사용 제한 없음) 미국 광상 기록 — 광상 유형·광체 모양·맥석·모암·구조·변질·성인·광화 규제·지체구조 항목을 이어 붙인 영어 문장, 광종 이름은 *** 로 가림

### 데이터 예시

`02_train.csv`

| record | commodity |
|---|---|
| dep_type: Vein | orebody_fm: IRREGULAR | hrock_type: Dacite | arock_type: Latite | Gold |
| hrock_type: Granite | arock_type: Granite | structure: Front Range, Tarryall Mountains, Lake George Ring Dike Complex | tectonic: Central Cordillera (Southern Rocky Mountains) | Uranium |
| orebody_fm: TABULAR | hrock_type: Limestone | structure: Fissures | conc_proc: Hydrothermal Fluids In Open Space Fillings. | ore_ctrl: Fissures | Gold |

`02_test_x.csv`

| id | record |
|---|---|
| 1 | dep_type: Metamorphic, Hydrothermal | orebody_fm: IRREGULAR | gangue: Graphite, Muscovite, Tourmaline, Quartz | hrock_type: Pegmatite |
| 2 | dep_type: Vein | orebody_fm: TABULAR, EN ECHELON | gangue: Beryl, Biotite, Hornblende, Muscovite, Orthoclase, Quartz | hrock_type: Hornblendite | arock_type: Pegmatite | alteration: Oxidized At Least 40 Feet Below Out Crop. | ore_ctrl: Faulting |
| 3 | gangue: Quartz | hrock_type: Greenstone | arock_type: Meta-Basalt | structure: Virgilina Synclinoriom | ore_ctrl: Shear Zones | tectonic: Piedmont |

### 문제 해결 가이드

(1) 훈련 데이터셋을 이용하여 광상 조사 기록으로 주 광종 분류(6종) 모델을 만드세요.
- 필요 시 추가로 함수를 만들거나 cell 을 만들어도 됩니다.
- 성능이 가장 좋은 모델을 저장하세요.
- 파일명 : **answer_2.pkl** (또는 다른 확장자의 모델 파일)

(2) 라벨('commodity') 값이 제외된 테스트 데이터 파일을 읽어들여 예측값에 대한 컬럼을 추가하여 csv 로 저장하세요.
- 파일명 : **answer_2.csv**
- 테스트 데이터 파일을 읽고 **마지막 컬럼 뒤에 'commodity' 컬럼 생성** 후, 모델 예측하여 예측값을 채워주세요.
- 테스트 데이터의 행 순서와 행 수를 바꾸지 마세요. 예측값은 훈련 데이터의 'commodity' 와 같은 형식(원래 라벨 이름·단위)이어야 합니다.

(3) 본 문제지를 답안코드 작성된 상태로 저장하세요.
- 파일명 : 시스템매핑번호.ipynb (본 문제지 상단의 저장버튼을 눌러 저장)

### 참고

- 영어 문장입니다. 단어 단위 `TfidfVectorizer(ngram_range=(1, 2))` 와 문자 n-gram 을 비교해 보세요. 항목 이름(`gangue:` 등)도 단서가 됩니다.
- 학습 라벨 분포: Gold 960, Uranium 960, Lead-Zinc 960, Copper 960, Iron 750, Tungsten 600 — 철·텅스텐이 조금 적습니다.
- 평가 지표는 **정확도(Accuracy)** 입니다. 테스트 행 순서(`id`)를 그대로 유지해 제출하세요.

### 채점 진행 방식 안내

- 2번 문항 배점 : 35점
- 시스템매핑번호.ipynb 실행 가능 여부 확인
- 제출한 코드로 실행했을 때, 제출한 모델 파일과 동일하게 재현되는지 확인
- answer_2.csv 로 제출된 예측 데이터를 정답지와 대조하여, **정확도(Accuracy) 53.1% 이상** 달성
  (공식 샘플의 '목표 70% → 채점 기준 63%' 와 같은 비율)

### 답안 작성

필요한 cell 을 추가하여 작성하세요.

In [ ]:
# 답안 작성

In [ ]:

import pandas as pd

# 데이터 불러오기
train_2 = pd.read_csv("02_train.csv")
test_2 = pd.read_csv("02_test_x.csv")

TEXT_2 = "record"
LABEL_2 = "commodity"

print("학습 데이터:", train_2.shape)
print("테스트 데이터:", test_2.shape)

print("\n컬럼 및 자료형")
train_2.info()
test_2.info()

print("\n결측치 개수")
print("학습:", train_2.isna().sum().sum())
print("테스트:", test_2.isna().sum().sum())

print("\n중복 텍스트 개수")
print("학습:", train_2[TEXT_2].duplicated().sum())
print("테스트:", test_2[TEXT_2].duplicated().sum())

print("\n광종별 데이터 개수")
print(train_2[LABEL_2].value_counts())

print("\n광종별 데이터 비율 (%)")
print((train_2[LABEL_2].value_counts(normalize=True) * 100).round(2))

print("\n텍스트 길이 통계")
print(train_2[TEXT_2].str.len().describe().round(1))

print("\n학습 데이터 예시")
display(train_2.head(3))

print("\n테스트 데이터 예시")
display(test_2.head(3))


학습 데이터: (5190, 2)
테스트 데이터: (1298, 2)

컬럼 및 자료형
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5190 entries, 0 to 5189
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   record     5190 non-null   object
 1   commodity  5190 non-null   object
dtypes: object(2)
memory usage: 81.2+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1298 entries, 0 to 1297
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      1298 non-null   int64 
 1   record  1298 non-null   object
dtypes: int64(1), object(1)
memory usage: 20.4+ KB

결측치 개수
학습: 0
테스트: 0

중복 텍스트 개수
학습: 0
테스트: 0

광종별 데이터 개수
commodity
Gold         960
Uranium      960
Lead-Zinc    960
Copper       960
Iron         750
Tungsten     600
Name: count, dtype: int64

광종별 데이터 비율 (%)
commodity
Gold         18.50
Uranium      18.50
Lead-Zinc    18.50
Copper       18.50
Iron         14.45
Tungsten     11.56
Name: proporti

,record,commodity
0,dep_type: Vein | orebody_fm: IRREGULAR | hrock...,Gold
1,hrock_type: Granite | arock_type: Granite | st...,Uranium
2,orebody_fm: TABULAR | hrock_type: Limestone | ...,Gold



테스트 데이터 예시


,id,record
0,1,"dep_type: Metamorphic, Hydrothermal | orebody_..."
1,2,"dep_type: Vein | orebody_fm: TABULAR, EN ECHEL..."
2,3,gangue: Quartz | hrock_type: Greenstone | aroc...


In [ ]:

from sklearn.model_selection import train_test_split
import pandas as pd

# 입력 텍스트와 정답 분리
X_2 = train_2[TEXT_2].copy()
y_2 = train_2[LABEL_2].copy()
X_test_2 = test_2[TEXT_2].copy()

# 학습 80%, 검증 20% 분리
X_train_2, X_valid_2, y_train_2, y_valid_2 = train_test_split(
    X_2,
    y_2,
    test_size=0.2,
    random_state=42,
    stratify=y_2
)

print("학습 X:", X_train_2.shape)
print("검증 X:", X_valid_2.shape)
print("학습 y:", y_train_2.shape)
print("검증 y:", y_valid_2.shape)
print("테스트 X:", X_test_2.shape)

# 학습·검증 클래스 비율 비교
class_ratio_2 = pd.DataFrame({
    "학습 (%)": y_train_2.value_counts(normalize=True) * 100,
    "검증 (%)": y_valid_2.value_counts(normalize=True) * 100
})

print("\n광종별 클래스 비율 비교")
display(class_ratio_2.round(2))

print("\n빈 텍스트 개수")
print("학습:", X_train_2.str.strip().eq("").sum())
print("검증:", X_valid_2.str.strip().eq("").sum())
print("테스트:", X_test_2.str.strip().eq("").sum())


학습 X: (4152,)
검증 X: (1038,)
학습 y: (4152,)
검증 y: (1038,)
테스트 X: (1298,)

광종별 클래스 비율 비교


,학습 (%),검증 (%)
commodity,,
Copper,18.50,18.50
Gold,18.50,18.50
Iron,14.45,14.45
Lead-Zinc,18.50,18.50
Tungsten,11.56,11.56
Uranium,18.50,18.50



빈 텍스트 개수
학습: 0
검증: 0
테스트: 0


In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score
import pandas as pd

# 4-Fold 교차검증
cv_2 = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

# TF-IDF 설정
word_tfidf_2 = {
    "analyzer": "word",
    "ngram_range": (1, 2),
    "min_df": 2,
    "sublinear_tf": True,
    "max_features": 60000
}

char_tfidf_2 = {
    "analyzer": "char_wb",
    "ngram_range": (2, 5),
    "min_df": 2,
    "sublinear_tf": True,
    "max_features": 80000
}

# 후보 모델 4개
models_2 = {
    "Word + LinearSVC": Pipeline([
        ("tfidf", TfidfVectorizer(**word_tfidf_2)),
        ("model", LinearSVC(C=1.0, random_state=42))
    ]),
    "Char + LinearSVC": Pipeline([
        ("tfidf", TfidfVectorizer(**char_tfidf_2)),
        ("model", LinearSVC(C=1.0, random_state=42))
    ]),
    "Word + LogisticRegression": Pipeline([
        ("tfidf", TfidfVectorizer(**word_tfidf_2)),
        ("model", LogisticRegression(max_iter=2000, random_state=42))
    ]),
    "Word + MultinomialNB": Pipeline([
        ("tfidf", TfidfVectorizer(**word_tfidf_2)),
        ("model", MultinomialNB(alpha=1.0))
    ])
}

# 교차검증 및 독립 검증
results_2 = []
fitted_models_2 = {}

for name, pipeline in models_2.items():
    cv_scores = cross_val_score(
        pipeline,
        X_train_2,
        y_train_2,
        cv=cv_2,
        scoring="accuracy",
        n_jobs=1
    )

    pipeline.fit(X_train_2, y_train_2)

    valid_pred = pipeline.predict(X_valid_2)
    valid_acc = accuracy_score(y_valid_2, valid_pred)

    fitted_models_2[name] = pipeline

    results_2.append({
        "모델": name,
        "CV Accuracy": cv_scores.mean(),
        "CV 표준편차": cv_scores.std(),
        "검증 Accuracy": valid_acc
    })

# 모델 성능 비교
result_df_2 = pd.DataFrame(results_2)
result_df_2 = result_df_2.sort_values(
    "CV Accuracy",
    ascending=False
).reset_index(drop=True)

display(result_df_2.round(4))


,모델,CV Accuracy,CV 표준편차,검증 Accuracy
0,Word + LogisticRegression,0.6344,0.0071,0.6464
1,Word + LinearSVC,0.6310,0.0154,0.6513
2,Char + LinearSVC,0.6286,0.0169,0.6175
3,Word + MultinomialNB,0.6014,0.0156,0.6243


In [ ]:

import joblib
import numpy as np
from sklearn.base import clone
from pathlib import Path

# 최종 모델 선택
best_model_name_2 = "Word + LogisticRegression"

# 기존 모델 설정 복제
final_model_2 = clone(fitted_models_2[best_model_name_2])

# 전체 5,190건으로 재학습
final_model_2.fit(X_2, y_2)

# 후보 모델 저장
model_path_2 = "answer_2_lr_candidate.pkl"
joblib.dump(final_model_2, model_path_2)

# 저장된 모델 다시 불러오기
loaded_model_2 = joblib.load(model_path_2)

# 저장 전후 테스트 예측 비교
pred_before_2 = final_model_2.predict(X_test_2)
pred_after_2 = loaded_model_2.predict(X_test_2)

print("선택 모델:", best_model_name_2)
print("모델 파일:", model_path_2)
print("파일 존재:", Path(model_path_2).exists())
print("전체 학습 건수:", len(X_2))
print("테스트 예측 건수:", len(pred_after_2))
print("저장 전후 예측 일치:", np.array_equal(
    pred_before_2, pred_after_2
))
print("예측 클래스:", np.unique(pred_after_2))
print("TF-IDF 특성 수:", len(
    loaded_model_2.named_steps["tfidf"].get_feature_names_out()
))


선택 모델: Word + LogisticRegression
모델 파일: answer_2_lr_candidate.pkl
파일 존재: True
전체 학습 건수: 5190
테스트 예측 건수: 1298
저장 전후 예측 일치: True
예측 클래스: ['Copper' 'Gold' 'Iron' 'Lead-Zinc' 'Tungsten' 'Uranium']
TF-IDF 특성 수: 14406


In [ ]:

import shutil
import hashlib
from pathlib import Path

# 정식 제출 파일명
final_model_path_2 = Path("answer_2.pkl")
final_csv_path_2 = Path("answer_2.csv")

# 기존 정식 제출 파일 보호
if final_model_path_2.exists() or final_csv_path_2.exists():
    raise FileExistsError("기존 2번 제출 파일이 있습니다. 덮어쓰기 전에 확인하세요.")

# 테스트 데이터에 예측 컬럼 추가
submission_2 = test_2.copy()
submission_2["commodity"] = loaded_model_2.predict(X_test_2)

# 제출 전 검증
assert len(submission_2) == len(test_2)
assert submission_2.columns.tolist() == test_2.columns.tolist() + ["commodity"]
assert submission_2["id"].equals(test_2["id"])
assert submission_2["commodity"].notna().all()
assert set(submission_2["commodity"]).issubset(set(y_2))

# 검증된 후보 모델을 정식 파일로 복사
shutil.copyfile(model_path_2, final_model_path_2)

# 예측 CSV 저장
submission_2.to_csv(final_csv_path_2, index=False)

# 저장된 파일 다시 확인
saved_csv_2 = pd.read_csv(final_csv_path_2)
saved_model_2 = joblib.load(final_model_path_2)

model_match_2 = (
    hashlib.sha256(Path(model_path_2).read_bytes()).hexdigest()
    == hashlib.sha256(final_model_path_2.read_bytes()).hexdigest()
)

csv_match_2 = saved_csv_2.equals(submission_2)

pred_match_2 = np.array_equal(
    saved_model_2.predict(X_test_2),
    saved_csv_2["commodity"].to_numpy()
)

print("모델 파일:", final_model_path_2)
print("CSV 파일:", final_csv_path_2)
print("CSV 크기:", saved_csv_2.shape)
print("마지막 컬럼:", saved_csv_2.columns[-1])
print("모델 파일 SHA-256 일치:", model_match_2)
print("CSV 저장 전후 일치:", csv_match_2)
print("모델 예측과 CSV 일치:", pred_match_2)

print("\n예측 클래스 분포")
print(saved_csv_2["commodity"].value_counts())

print("\n제출 CSV 상위 3행")
display(saved_csv_2.head(3))


모델 파일: answer_2.pkl
CSV 파일: answer_2.csv
CSV 크기: (1298, 3)
마지막 컬럼: commodity
모델 파일 SHA-256 일치: True
CSV 저장 전후 일치: True
모델 예측과 CSV 일치: True

예측 클래스 분포
commodity
Lead-Zinc    268
Gold         253
Uranium      234
Copper       228
Iron         162
Tungsten     153
Name: count, dtype: int64

제출 CSV 상위 3행


,id,record,commodity
0,1,"dep_type: Metamorphic, Hydrothermal | orebody_...",Uranium
1,2,"dep_type: Vein | orebody_fm: TABULAR, EN ECHEL...",Gold
2,3,gangue: Quartz | hrock_type: Greenstone | aroc...,Copper


<details><summary>힌트 1단계 (접근 방향) - 2번 문항</summary>

라벨 분포와 문장 길이를 확인합니다. 문장을 TF-IDF 로 바꾸되 **단어 단위와 글자 조각(char n-gram)** 을 비교하고, 나이브 베이즈·선형 SVM·로지스틱 회귀 중 **교차검증 정확도** 가 높은 조합으로 전체 학습하세요. 다른 컬럼(숫자·범주)이 단서이면 `ColumnTransformer` 로 함께 넣습니다.

- 이 문항의 핵심: 영어 문장입니다. 단어 단위 `TfidfVectorizer(ngram_range=(1, 2))` 와 문자 n-gram 을 비교해 보세요. 항목 이름(`gangue:` 등)도 단서가 됩니다.

</details>

<details><summary>힌트 2단계 (사용할 도구) - 2번 문항</summary>

사용할 도구: `pandas`, `joblib`, `make_pipeline`, `CountVectorizer`, `TfidfVectorizer`, `MultinomialNB`, `LinearSVC`, `train_test_split`, `accuracy_score`.

- 참고 문서: <https://joblib.readthedocs.io/en/stable/generated/joblib.dump.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html>
- 참고 문서: <https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html>

</details>

<details><summary>힌트 3단계 (코드 구조) - 2번 문항</summary>

코드의 **앞부분은 구조만** 보여 줍니다(`____` 자리는 가이드와 공식 문서를 보고 채우세요). **뒷부분은 주석으로 할 일만** 적었습니다 — 주석을 따라 직접 작성하세요.
```python
import ____
import ____
import ____
from sklearn.pipeline import ____
from sklearn.feature_extraction.text import ____, ____
from sklearn.naive_bayes import ____
from sklearn.svm import ____
from sklearn.model_selection import ____
from sklearn.metrics import ____

train = pd.read_csv('02_train.csv')
test = pd.read_csv('02_test_x.csv')
TEXT, LABEL = '____', '____'
print(train[LABEL].value_counts().to_dict(), '____', round(train[TEXT].str.len().mean(), ____))


# ▷ 학습(검증 데이터·조기 종료)
# ▷ 모델 파일 저장 — answer_N.pkl / .h5
# ▷ 테스트 데이터 예측(행 순서 그대로)
# ▷ 예측 CSV 저장 — answer_N.csv(테스트 열 + 마지막 정답 열)
```
- 제출 파일명: `answer_2.pkl`(모델) · `answer_2.csv`(예측)

</details>

## 3. Image 문제 - 중력지도 조각으로 지형구 분류(6종) (35점)

<sub>학습 목표: 중력지도 조각으로 지형구 분류(6종) 모델로 테스트 정확도 0.58 이상을 달성할 수 있다 | 역량: PI1 이미지 로딩·전처리, PI2 이미지 분류 모델(CNN·전이학습) | 기술: pandas, Keras(TensorFlow.js) | 난이도: 고급 | 예상 시간: 60분</sub>

### 문제 개요

**중력지도(부게 이상) 조각** 을 보고 그 지역이 어느 **지형구(physiographic division) 6종** 에 속하는지 분류하는 이미지 분류 문제입니다. 산계는 지각 뿌리 때문에 큰 음(−)의 이상(파랑)이, 연안 평야·애팔래치아는 0 근처(노랑·빨강)의 줄무늬 이상이 나타나는 등 지역마다 무늬가 다릅니다. 학습·테스트 조각은 **서로 다른 지역 블록** 에서 뽑아, 겹치는 조각이 없습니다.

제공된 훈련 데이터셋을 활용해 이미지 다중 분류(6종) 모델을 개발하고, 테스트 데이터셋의 'label' 값을 예측하시기 바랍니다.

### 달성 목표

"03_test_x.csv" 파일의 테스트 데이터셋을 기준으로 **정확도(Accuracy) 58% 이상**을 달성하는 모델을 개발하시기 바랍니다.

단, 테스트 데이터의 정답(Label 값)은 별도로 제공되지 않습니다.

### 데이터 설명

**데이터 위치 및 폴더 구조** — 문제 파일과 동일한 곳에 위치

- `03_images.zip`: 압축을 풀면 `03_train/<지형구>/grav_*.png`(학습, 클래스별 폴더)와 `03_test/test_*.png`(테스트) 폴더가 생깁니다. 64×64 컬러 PNG, 북쪽이 위.
- `03_train.csv`: 학습 이미지 파일명과 라벨
- `03_test_x.csv`: 테스트 이미지 파일명(라벨 제외)

**Label 정보**

- 'label' : appalachian_highlands: 애팔래치아 고지 / atlantic_plain: 대서양 연안 평야 / interior_plains: 내륙 평원 / intermontane_plateaus: 산간 고원(분지·산맥 등) / pacific_mountain: 태평양 산계 / rocky_mountain: 로키 산계

**데이터 출처** — USGS 미국 본토 완전 부게 중력 이상 격자(미국 정부 공공 저작물)를 약 96km × 96km 조각으로 잘라 색 지도(jet, −250~50 mGal 고정)로 그린 64×64 PNG, 라벨은 USGS 미국 지형구(Fenneman & Johnson 1946) 대구분 6종 — 조각의 90% 이상이 한 지형구인 것만

### 데이터 예시

학습 이미지 라벨별 장수: interior_plains 330, atlantic_plain 330, intermontane_plateaus 330, appalachian_highlands 330, rocky_mountain 297, pacific_mountain 203

| file_name | label |
|---|---|
| grav_00001.png | interior_plains |
| grav_00002.png | atlantic_plain |
| grav_00003.png | pacific_mountain |

### 문제 해결 가이드

(1) 훈련 데이터셋을 이용하여 중력지도 조각으로 지형구 분류(6종) 모델을 만드세요.
- 필요 시 추가로 함수를 만들거나 cell 을 만들어도 됩니다.
- 성능이 가장 좋은 모델을 저장하세요.
- 파일명 : **answer_3.h5** (또는 다른 확장자의 모델 파일)

(2) 라벨('label') 값이 제외된 테스트 데이터 파일을 읽어들여 예측값에 대한 컬럼을 추가하여 csv 로 저장하세요.
- 파일명 : **answer_3.csv**
- 테스트 데이터 파일을 읽고 **마지막 컬럼 뒤에 'label' 컬럼 생성** 후, 모델 예측하여 예측값을 채워주세요.
- 테스트 데이터의 행 순서와 행 수를 바꾸지 마세요. 예측값은 훈련 데이터의 'label' 와 같은 형식(원래 라벨 이름·단위)이어야 합니다.

(3) 본 문제지를 답안코드 작성된 상태로 저장하세요.
- 파일명 : 시스템매핑번호.ipynb (본 문제지 상단의 저장버튼을 눌러 저장)

### 참고

- 색은 모든 조각에 같은 기준(−250 mGal = 진한 파랑, 50 mGal = 진한 빨강)으로 칠했습니다. 색 자체가 중력 이상의 크기입니다.
- 학습 장수가 지형구마다 다릅니다(태평양 산계·로키 산계가 적음). 좌우·상하 뒤집기 증강으로 학습 데이터를 늘려 보세요.
- 평가 지표는 **정확도(Accuracy)** 입니다. 테스트는 `03_test_x.csv` 의 파일명 순서대로 읽어 예측하세요.

### 채점 진행 방식 안내

- 3번 문항 배점 : 35점
- 시스템매핑번호.ipynb 실행 가능 여부 확인
- 제출한 코드로 실행했을 때, 제출한 모델 파일과 동일하게 재현되는지 확인
- answer_3.csv 로 제출된 예측 데이터를 정답지와 대조하여, **정확도(Accuracy) 52.2% 이상** 달성
  (공식 샘플의 '목표 70% → 채점 기준 63%' 와 같은 비율)

### 답안 작성

필요한 cell 을 추가하여 작성하세요.

In [ ]:
# 답안 작성

In [ ]:

import os
import zipfile
from pathlib import Path
from collections import Counter
import pandas as pd
from PIL import Image

# 이미지 압축 해제
IMG_DIR_3 = Path(os.environ.get("AICE_IMG_DIR", "aice_images_10"))

train_dir_3 = IMG_DIR_3 / "03_train"
test_dir_3 = IMG_DIR_3 / "03_test"

if not train_dir_3.exists() or not test_dir_3.exists():
    with zipfile.ZipFile("03_images.zip") as z:
        z.extractall(IMG_DIR_3)

# CSV 불러오기
train_3 = pd.read_csv("03_train.csv")
test_3 = pd.read_csv("03_test_x.csv")

print("학습 CSV:", train_3.shape)
print("테스트 CSV:", test_3.shape)

print("\n결측치 개수")
print("학습:", train_3.isna().sum().sum())
print("테스트:", test_3.isna().sum().sum())

print("\n클래스별 학습 이미지 수")
print(train_3["label"].value_counts())

# CSV 파일명과 실제 이미지 경로 연결
train_paths_3 = [
    train_dir_3 / label / file_name
    for file_name, label in zip(
        train_3["file_name"], train_3["label"]
    )
]

test_paths_3 = [
    test_dir_3 / file_name
    for file_name in test_3["file_name"]
]

missing_train_3 = sum(not p.is_file() for p in train_paths_3)
missing_test_3 = sum(not p.is_file() for p in test_paths_3)

print("\n이미지 파일 확인")
print("학습 파일 누락:", missing_train_3)
print("테스트 파일 누락:", missing_test_3)

# 이미지 크기와 채널 확인
image_info_3 = Counter()

for path in train_paths_3 + test_paths_3:
    if path.is_file():
        with Image.open(path) as img:
            image_info_3[(img.size, img.mode)] += 1

print("\n이미지 크기 및 채널별 개수")
print(dict(image_info_3))

print("\n학습 데이터 예시")
display(train_3.head(3))

print("\n테스트 데이터 예시")
display(test_3.head(3))


학습 CSV: (1820, 2)
테스트 CSV: (382, 1)

결측치 개수
학습: 0
테스트: 0

클래스별 학습 이미지 수
label
interior_plains          330
atlantic_plain           330
appalachian_highlands    330
intermontane_plateaus    330
rocky_mountain           297
pacific_mountain         203
Name: count, dtype: int64

이미지 파일 확인
학습 파일 누락: 0
테스트 파일 누락: 0

이미지 크기 및 채널별 개수
{((64, 64), 'RGB'): 2202}

학습 데이터 예시


,file_name,label
0,grav_00001.png,interior_plains
1,grav_00002.png,atlantic_plain
2,grav_00003.png,pacific_mountain



테스트 데이터 예시


,file_name
0,test_0001.png
1,test_0002.png
2,test_0003.png


In [ ]:

import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 라벨을 숫자로 변환
label_encoder_3 = LabelEncoder()
y_all_3 = label_encoder_3.fit_transform(train_3["label"])

# 이미지 불러오기
def load_image_3(path):
    with Image.open(path) as img:
        return np.asarray(img.convert("RGB"), dtype=np.uint8)

X_all_3 = np.stack([
    load_image_3(path) for path in train_paths_3
])

# 학습·검증 인덱스 분리
idx_train_3, idx_valid_3 = train_test_split(
    np.arange(len(y_all_3)),
    test_size=0.2,
    random_state=42,
    stratify=y_all_3
)

# 이미지 정규화 (0~1)
X_train_img_3 = X_all_3[idx_train_3].astype("float32") / 255.0
X_valid_img_3 = X_all_3[idx_valid_3].astype("float32") / 255.0

# 정답 라벨 분리
y_train_img_3 = y_all_3[idx_train_3]
y_valid_img_3 = y_all_3[idx_valid_3]

print("학습 이미지:", X_train_img_3.shape)
print("검증 이미지:", X_valid_img_3.shape)

print("\n학습 라벨:", y_train_img_3.shape)
print("검증 라벨:", y_valid_img_3.shape)

print("\n이미지 자료형:", X_train_img_3.dtype)
print("픽셀 최솟값:", X_train_img_3.min())
print("픽셀 최댓값:", X_train_img_3.max())

print("\n클래스 번호")
for i, name in enumerate(label_encoder_3.classes_):
    print(i, name)

print("\n클래스별 학습·검증 이미지 수")
for i, name in enumerate(label_encoder_3.classes_):
    train_count = np.sum(y_train_img_3 == i)
    valid_count = np.sum(y_valid_img_3 == i)
    print(f"{name}: 학습 {train_count}장 / 검증 {valid_count}장")


학습 이미지: (1456, 64, 64, 3)
검증 이미지: (364, 64, 64, 3)

학습 라벨: (1456,)
검증 라벨: (364,)

이미지 자료형: float32
픽셀 최솟값: 0.0
픽셀 최댓값: 1.0

클래스 번호
0 appalachian_highlands
1 atlantic_plain
2 interior_plains
3 intermontane_plateaus
4 pacific_mountain
5 rocky_mountain

클래스별 학습·검증 이미지 수
appalachian_highlands: 학습 264장 / 검증 66장
atlantic_plain: 학습 264장 / 검증 66장
interior_plains: 학습 264장 / 검증 66장
intermontane_plateaus: 학습 264장 / 검증 66장
pacific_mountain: 학습 162장 / 검증 41장
rocky_mountain: 학습 238장 / 검증 59장


In [ ]:

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input, Conv2D, BatchNormalization,
    MaxPooling2D, Dropout,
    GlobalAveragePooling2D, Dense
)

# 재현성을 위한 시드 설정
tf.random.set_seed(42)

# CNN 모델 구성
model_3 = Sequential([
    Input(shape=(64, 64, 3)),

    # Block 1
    Conv2D(24, 3, padding="same", activation="relu"),
    Conv2D(24, 3, padding="same", activation="relu"),
    BatchNormalization(),
    MaxPooling2D(),
    Dropout(0.15),

    # Block 2
    Conv2D(48, 3, padding="same", activation="relu"),
    Conv2D(48, 3, padding="same", activation="relu"),
    BatchNormalization(),
    MaxPooling2D(),
    Dropout(0.20),

    # Block 3
    Conv2D(96, 3, padding="same", activation="relu"),
    Conv2D(96, 3, padding="same", activation="relu"),
    BatchNormalization(),
    MaxPooling2D(),
    Dropout(0.25),

    # 분류기
    GlobalAveragePooling2D(),
    Dense(64, activation="relu"),
    Dropout(0.30),
    Dense(6, activation="softmax")
])

# 모델 구조 확인
model_3.summary()

print("\n입력 형태:", model_3.input_shape)
print("출력 형태:", model_3.output_shape)
print("전체 파라미터:", model_3.count_params())
print("클래스 수:", len(label_encoder_3.classes_))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 64, 64, 24)     │           672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 24)     │         5,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 64, 24)     │            96 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 24)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32, 32, 24)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 48)     │        10,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 48)     │        20,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 48)     │           192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 48)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16, 16, 48)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 16, 16, 96)     │        41,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 16, 16, 96)     │        83,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 16, 96)     │           384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 8, 96)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 8, 8, 96)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 96)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 168,958 (659.99 KB)

 Trainable params: 168,622 (658.68 KB)

 Non-trainable params: 336 (1.31 KB)


입력 형태: (None, 64, 64, 3)
출력 형태: (None, 6)
전체 파라미터: 168958
클래스 수: 6


In [ ]:

import tensorflow as tf
import numpy as np
from pathlib import Path
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)

# 최고 모델 저장 경로
checkpoint_path_3 = "answer_3_cnn_v1.keras"

# 기존 후보 모델 보호
if Path(checkpoint_path_3).exists():
    raise FileExistsError("기존 CNN 후보 모델이 있습니다. 덮어쓰지 않습니다.")

# 모델 컴파일
model_3.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# 최고 검증 Accuracy 모델 저장
checkpoint_3 = ModelCheckpoint(
    filepath=checkpoint_path_3,
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1
)

# 검증 손실 개선이 없으면 학습 중단
earlystop_3 = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True,
    verbose=1
)

# 학습률 자동 감소
reduce_lr_3 = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=0.00001,
    verbose=1
)

# CNN 학습
history_3 = model_3.fit(
    X_train_img_3,
    y_train_img_3,
    validation_data=(X_valid_img_3, y_valid_img_3),
    epochs=35,
    batch_size=32,
    shuffle=True,
    callbacks=[checkpoint_3, earlystop_3, reduce_lr_3],
    verbose=2
)

# 학습 결과 요약
best_epoch_3 = np.argmax(history_3.history["val_accuracy"]) + 1
best_acc_3 = max(history_3.history["val_accuracy"])

print("\n실제 학습 Epoch:", len(history_3.history["loss"]))
print("최고 검증 Epoch:", best_epoch_3)
print("최고 검증 Accuracy:", round(best_acc_3 * 100, 2), "%")
print("최고 모델 파일:", checkpoint_path_3)
print("파일 저장 여부:", Path(checkpoint_path_3).exists())


Epoch 1/35

Epoch 1: val_accuracy improved from None to 0.32967, saving model to answer_3_cnn_v1.keras

Epoch 1: finished saving model to answer_3_cnn_v1.keras
46/46 - 24s - 523ms/step - accuracy: 0.5048 - loss: 1.2264 - val_accuracy: 0.3297 - val_loss: 1.6276 - learning_rate: 0.0010
Epoch 2/35

Epoch 2: val_accuracy improved from 0.32967 to 0.42857, saving model to answer_3_cnn_v1.keras

Epoch 2: finished saving model to answer_3_cnn_v1.keras
46/46 - 19s - 409ms/step - accuracy: 0.5130 - loss: 1.0837 - val_accuracy: 0.4286 - val_loss: 1.4860 - learning_rate: 0.0010
Epoch 3/35

Epoch 3: val_accuracy improved from 0.42857 to 0.43407, saving model to answer_3_cnn_v1.keras

Epoch 3: finished saving model to answer_3_cnn_v1.keras
46/46 - 22s - 472ms/step - accuracy: 0.5591 - loss: 1.0199 - val_accuracy: 0.4341 - val_loss: 1.4771 - learning_rate: 0.0010
Epoch 4/35

Epoch 4: val_accuracy did not improve from 0.43407
46/46 - 20s - 429ms/step - accuracy: 0.5755 - loss: 0.9892 - val_accuracy: 0

In [ ]:

import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# 최고 검증 Accuracy 모델 불러오기
best_model_3 = tf.keras.models.load_model(
    "answer_3_cnn_v1.keras"
)

# 저장된 모델의 검증 성능 확인
valid_loss_3, valid_acc_3 = best_model_3.evaluate(
    X_valid_img_3,
    y_valid_img_3,
    verbose=0
)

# 검증 이미지 예측
valid_probs_3 = best_model_3.predict(
    X_valid_img_3,
    verbose=0
)

valid_pred_3 = np.argmax(valid_probs_3, axis=1)

print("저장 모델 검증 Loss:", round(valid_loss_3, 4))
print("저장 모델 검증 Accuracy:", round(valid_acc_3 * 100, 2), "%")
print("예측 행 수:", len(valid_pred_3))
print("정답 클래스 수:", len(label_encoder_3.classes_))

# 클래스별 성능
report_3 = classification_report(
    y_valid_img_3,
    valid_pred_3,
    labels=np.arange(6),
    target_names=label_encoder_3.classes_,
    output_dict=True,
    zero_division=0
)

report_df_3 = pd.DataFrame(report_3).T

print("\n클래스별 성능")
display(report_df_3.round(3))

# 혼동 행렬
cm_3 = confusion_matrix(
    y_valid_img_3,
    valid_pred_3,
    labels=np.arange(6)
)

cm_df_3 = pd.DataFrame(
    cm_3,
    index=label_encoder_3.classes_,
    columns=label_encoder_3.classes_
)

print("\n혼동 행렬 (행=실제, 열=예측)")
display(cm_df_3)


저장 모델 검증 Loss: 0.6102
저장 모델 검증 Accuracy: 78.02 %
예측 행 수: 364
정답 클래스 수: 6

클래스별 성능


,precision,recall,f1-score,support
appalachian_highlands,0.920,0.697,0.793,66.00
atlantic_plain,0.903,0.985,0.942,66.00
interior_plains,0.627,0.788,0.698,66.00
intermontane_plateaus,0.677,0.636,0.656,66.00
pacific_mountain,0.923,0.878,0.900,41.00
rocky_mountain,0.741,0.729,0.735,59.00
accuracy,0.780,0.780,0.780,0.78
macro avg,0.799,0.785,0.787,364.00
weighted avg,0.791,0.780,0.781,364.00



혼동 행렬 (행=실제, 열=예측)


,appalachian_highlands,atlantic_plain,interior_plains,intermontane_plateaus,pacific_mountain,rocky_mountain
appalachian_highlands,46,4,16,0,0,0
atlantic_plain,0,65,1,0,0,0
interior_plains,3,2,52,9,0,0
intermontane_plateaus,0,0,6,42,3,15
pacific_mountain,1,1,3,0,36,0
rocky_mountain,0,0,5,11,0,43


In [ ]:

import numpy as np
import pandas as pd
import tensorflow as tf
import shutil
import hashlib
from pathlib import Path

# 정식 제출 파일 경로
final_model_path_3 = Path("answer_3.keras")
final_csv_path_3 = Path("answer_3.csv")

# 기존 제출 파일 보호
if final_model_path_3.exists() or final_csv_path_3.exists():
    raise FileExistsError("기존 3번 제출 파일이 있습니다. 덮어쓰지 않습니다.")

# 테스트 이미지 382장 불러오기 및 정규화
X_test_img_3 = np.stack([
    load_image_3(path) for path in test_paths_3
]).astype("float32") / 255.0

# 저장된 최고 모델로 테스트 예측
test_probs_3 = best_model_3.predict(
    X_test_img_3,
    verbose=0
)

# 예측 확률 → 클래스 번호 → 실제 라벨 이름
test_pred_idx_3 = np.argmax(test_probs_3, axis=1)
test_pred_label_3 = label_encoder_3.inverse_transform(
    test_pred_idx_3
)

# 제출 CSV 생성
submission_3 = test_3.copy()
submission_3["label"] = test_pred_label_3

# 제출 전 검증
assert len(submission_3) == len(test_3)
assert submission_3.columns.tolist() == test_3.columns.tolist() + ["label"]
assert submission_3["file_name"].equals(test_3["file_name"])
assert submission_3["label"].notna().all()
assert set(submission_3["label"]).issubset(set(label_encoder_3.classes_))

# 최고 체크포인트를 정식 모델 파일로 복사
shutil.copyfile(checkpoint_path_3, final_model_path_3)

# CSV 저장
submission_3.to_csv(final_csv_path_3, index=False)

# 저장된 파일 재검증
saved_csv_3 = pd.read_csv(final_csv_path_3)
saved_model_3 = tf.keras.models.load_model(final_model_path_3)

saved_probs_3 = saved_model_3.predict(
    X_test_img_3,
    verbose=0
)

saved_labels_3 = label_encoder_3.inverse_transform(
    np.argmax(saved_probs_3, axis=1)
)

model_match_3 = (
    hashlib.sha256(Path(checkpoint_path_3).read_bytes()).hexdigest()
    == hashlib.sha256(final_model_path_3.read_bytes()).hexdigest()
)

csv_match_3 = saved_csv_3.equals(submission_3)
pred_match_3 = np.array_equal(
    saved_labels_3,
    saved_csv_3["label"].to_numpy()
)

print("모델 파일:", final_model_path_3)
print("CSV 파일:", final_csv_path_3)
print("CSV 크기:", saved_csv_3.shape)
print("마지막 컬럼:", saved_csv_3.columns[-1])
print("모델 파일 SHA-256 일치:", model_match_3)
print("CSV 저장 전후 일치:", csv_match_3)
print("모델 예측과 CSV 일치:", pred_match_3)

print("\n예측 클래스 분포")
print(saved_csv_3["label"].value_counts())

print("\n제출 CSV 상위 3행")
display(saved_csv_3.head(3))


모델 파일: answer_3.keras
CSV 파일: answer_3.csv
CSV 크기: (382, 2)
마지막 컬럼: label
모델 파일 SHA-256 일치: True
CSV 저장 전후 일치: True
모델 예측과 CSV 일치: True

예측 클래스 분포
label
interior_plains          97
atlantic_plain           92
intermontane_plateaus    56
pacific_mountain         52
appalachian_highlands    47
rocky_mountain           38
Name: count, dtype: int64

제출 CSV 상위 3행


,file_name,label
0,test_0001.png,atlantic_plain
1,test_0002.png,appalachian_highlands
2,test_0003.png,interior_plains


<details><summary>힌트 1단계 (접근 방향) - 3번 문항</summary>

압축을 풀고 폴더 구조(학습: 클래스별 폴더, 테스트: 한 폴더)를 확인합니다. 이미지를 같은 크기로 읽어 0~1 로 맞추고, 검증 데이터를 떼어 **작은 CNN**(또는 사전학습 모델 특징 + 분류기)을 학습합니다. 증강·Dropout·조기 종료로 과적합을 막고, 테스트는 `03_test_x.csv` 순서대로 예측하세요.

- 이 문항의 핵심: 색은 모든 조각에 같은 기준(−250 mGal = 진한 파랑, 50 mGal = 진한 빨강)으로 칠했습니다. 색 자체가 중력 이상의 크기입니다.

</details>

<details><summary>힌트 2단계 (사용할 도구) - 3번 문항</summary>

사용할 도구: `tensorflow`, `pandas`, `image_dataset_from_directory`, `load_img`, `img_to_array`, `Sequential`, `load_model`, `Input`, `Conv2D`, `MaxPooling2D`, `Flatten`, `Dense`, `Dropout`, `EarlyStopping`.

- 참고 문서: <https://keras.io/api/data_loading/image/>
- 참고 문서: <https://keras.io/api/data_loading/image/#loadimg-function>
- 참고 문서: <https://keras.io/api/data_loading/image/#imgtoarray-function>
- 참고 문서: <https://keras.io/api/models/sequential/>
- 참고 문서: <https://keras.io/api/layers/convolution_layers/convolution2d/>
- 참고 문서: <https://keras.io/api/layers/pooling_layers/max_pooling2d/>

</details>

<details><summary>힌트 3단계 (코드 구조) - 3번 문항</summary>

코드의 **앞부분은 구조만** 보여 줍니다(`____` 자리는 가이드와 공식 문서를 보고 채우세요). **뒷부분은 주석으로 할 일만** 적었습니다 — 주석을 따라 직접 작성하세요.
```python
import ____
import ____
IMG_DIR = os.environ.get('____', '____')        # 압축을 풀 위치(브라우저판은 메모리 폴더 /tmp)
if not os.path.exists(os.path.join(IMG_DIR, '____')):
    zipfile.ZipFile('03_images.zip').extractall(IMG_DIR)
import ____
from tensorflow.keras.utils import ____, ____, ____
from tensorflow.keras.models import ____, ____
from tensorflow.keras.layers import ____, ____, ____, ____, ____, ____
from tensorflow.keras.callbacks import ____, ____, ____

train_csv = pd.read_csv('03_train.csv')
test = pd.read_csv('03_test_x.csv')
kw = dict(image_size=(____, ____), batch_size=____, validation_split=____, seed=____)
tr_ds = image_dataset_from_directory(f'____', subset='____', **kw)
va_ds = image_dataset_from_directory(f'____', subset='____', **kw)
CLASSES = tr_ds.class_names                               # 폴더 이름(알파벳 순) = 라벨 번호 순서

def to_numpy(ds):                                         # 배치를 모아 배열로(____~____ 정규화)
    xs, ys = zip(*[(np.asarray(x), np.asarray(y)) for x, y in ds])
    return np.concatenate(xs).astype('____') / ____, np.concatenate(ys)

# ▷ 모델 구성(CNN 또는 사전학습 특징 추출)
# ▷ 학습(검증 데이터·조기 종료)
# ▷ 모델 파일 저장 — answer_N.pkl / .h5
# ▷ 테스트 데이터 예측(행 순서 그대로)
# ▷ 예측 CSV 저장 — answer_N.csv(테스트 열 + 마지막 정답 열)
```
- 제출 파일명: `answer_3.h5`(모델) · `answer_3.csv`(예측)

</details>